# Price Of Ethereum — quickstart

Measure real onchain depth from your own Fynd, understand what a measurement
contains, then record a short history.

**Prerequisites:** a local Fynd (`fynd serve --chain ethereum
--worker-pools-config worker_pools.toml`) and `TYCHO_API_KEY` in the
environment. Install with the viz extra: `pip install "price-of-ethereum[viz]"`.

Every number below is a Fynd quote or a documented function of quotes. No
oracles, no estimates.

## 1. Connect

Tycho is the only metadata source — decimals, symbol, quality tier, transfer
tax. There is no RPC client anywhere in this package.

In [ ]:
import os

import pandas as pd

from price_of_ethereum import FyndClient, TychoClient, resolve_tokens

WETH = "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2"
USDC = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"

fynd = FyndClient("http://127.0.0.1:3000")
fynd.wait_until_ready()  # cold-start hydration takes 1-5 minutes
chain_id = fynd.info().chain_id

tycho = TychoClient("https://tycho-beta.propellerheads.xyz", os.environ["TYCHO_API_KEY"])
tokens = resolve_tokens(tycho, [WETH, USDC])
chain_id, tokens[WETH.lower()], tokens[USDC.lower()]

## 2. Take one snapshot

One spot probe, then a two-sided log-spaced sweep, then anchored bisections at
the headline impact levels. `mid_source` reports how the mid was won —
`sweep_band` is the healthy path; `probe_fallback` and `spot_degraded` mean thin
liquidity or failing quotes.

`samples_per_side` defaults to 100. It is **not** a free resolution knob: it
decides which rungs land in the robust-mid depth band, so it is part of the
mid's definition. Lowering it here only to explore faster.

In [ ]:
from price_of_ethereum import SnapshotConfig, collect_snapshot

config = SnapshotConfig(
    token=tokens[WETH.lower()],
    numeraire=tokens[USDC.lower()],
    pair="ETH/USDC",
    chain_id=chain_id,
    samples_per_side=40,
    search_max=5_000_000,
)
snapshot = collect_snapshot(fynd, config)
snapshot.to_block_row()

## 3. What a measurement contains

`curve` rows are sweep rungs; `anchor` rows are bisected headline levels solved
with `min_responses=0` so the split-routing solver contributes.

In [ ]:
rows = pd.DataFrame.from_records(snapshot.to_rows())
blocks = pd.DataFrame.from_records([snapshot.to_block_row()])
rows.groupby(["kind", "side"]).size()

In [ ]:
rows[rows["kind"] == "curve"].head(3).T

`impact_pct` is measured against `spot` (the $1,000 probe). `price_impact_bps`
is derived against `robust_mid`. `price_impact_bps_raw` is whatever Fynd
reported — nullable, stored for reference, never used for control flow.

In [ ]:
level_columns = [
    "side",
    "target_impact_pct",
    "size_numeraire",
    "impact_pct",
    "price_impact_bps",
    "price_impact_bps_raw",
    "bound",
    "target_reached",
    "derived_from",
]
rows.loc[rows["kind"] == "anchor", level_columns]

## 4. Charts

These are the same figure builders `poe serve` uses, so a notebook draws exactly
what the dashboard draws.

In [ ]:
from price_of_ethereum.dashboard import book_map_figure, cost_curve_figure, spread_curve_figure

cost_curve_figure(rows).show()

Diamonds are anchored measurements; the line is the bulk sweep. The shaded band
in the book map below is the depth range the robust mid is voted from.

In [ ]:
book_map_figure(rows, snapshot.robust_mid).show()
spread_curve_figure(rows, snapshot.robust_mid).show()

## 5. Where impact stops being monotonic

Measured impact does not have to rise with size: as size grows the router can
recompose the route across more pools and impact can dip. Each negative
difference below is a real routing change, not noise. (Against a single-pool
simulator this comes back empty; against real liquidity it usually does not.)

In [ ]:
curve_buy = rows[(rows["kind"] == "curve") & (rows["side"] == "buy")]
buy = curve_buy.sort_values("size_numeraire").copy()
buy["impact_delta"] = buy["impact_pct"].diff()
buy.loc[
    buy["impact_delta"] < 0,
    ["size_numeraire", "impact_pct", "impact_delta", "n_pools", "route_hash"],
]

## 6. Record a history

`collect_blocks` detects a new block by comparing each snapshot's majority block
to the last recorded one. Rows are appended before the block summary, so the
blocks file is the index of blocks whose rows are fully on disk — join against
it.

In [ ]:
from price_of_ethereum.collect import collect_blocks

result = collect_blocks(fynd, config, out_dir="data", blocks=10)
result

In [ ]:
from price_of_ethereum import load_jsonl, load_parquet, to_parquet
from price_of_ethereum.dashboard import history_health_figure, history_mid_figure

recorded = load_jsonl(result.blocks_path)
summary_columns = [
    "block_number",
    "spot",
    "robust_mid",
    "median_depth",
    "mid_source",
    "mixed_block",
    "duration_ms",
]
recorded[summary_columns]

In [ ]:
history_mid_figure(recorded).show()
history_health_figure(recorded).show()  # bars colored where mixed_block fired

JSONL is the collection format; parquet is the analysis format.

In [ ]:
all_rows = load_jsonl(result.rows_path)
to_parquet(all_rows, "data/eth-usdc_1.rows.parquet")
load_parquet("data/eth-usdc_1.rows.parquet").groupby(["block_number", "kind"]).size().head()

## 7. Or watch it live

For ongoing exploration, run the collector and the dashboard side by side
instead of re-running this notebook:

```bash
poe collect --out data &                      # keep measuring
poe serve --out data                          # http://127.0.0.1:8765
poe report --out data --output report.html    # frozen, self-contained copy
```

The dashboard shows the cost curve, book map, round-trip spread, the anchored
level table, and mid / depth / latency across every recorded block. It reads
only what is on disk and contacts nothing.